# 3. Sentiment Classification

In [1]:
import os, sys
try:
    _HERE = os.path.dirname(os.path.abspath(__file__))
except NameError:
    _HERE = os.getcwd()
for _c in (_HERE, os.path.dirname(_HERE)):
    _p = os.path.join(_c, "app")
    if os.path.isdir(_p):
        sys.path.insert(0, _p); break

import joblib, numpy as np, pandas as pd
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (GridSearchCV, StratifiedKFold,
                                     cross_val_predict, train_test_split)
from sklearn.metrics import classification_report, mean_absolute_error, f1_score, accuracy_score
from text_utils import (CLEAN_CSV, MODELS_DIR, LexiconFeatures,
                        clean_review_text, probs_to_outputs, rating_to_sentiment)

df = pd.read_csv(CLEAN_CSV)
X = [str(t) for t in df["review_clean"].tolist()]   # ← plain Python strings (the fix)
y = df["Rating"].astype(int).to_numpy()             # ← plain numpy ints
print("reviews:", len(df))
print(pd.Series(y).value_counts().sort_index())

reviews: 587
1     59
2     36
3     53
4    144
5    295
Name: count, dtype: int64


In [2]:
def build_pipeline():
    features = FeatureUnion([
        ("w1", TfidfVectorizer(ngram_range=(1, 1), min_df=1, stop_words=None,
                               sublinear_tf=True, token_pattern=r"(?u)\b[\w']+\b")),
        ("w2", TfidfVectorizer(ngram_range=(2, 2), min_df=2, stop_words=None,
                               sublinear_tf=True, token_pattern=r"(?u)\b[\w']+\b")),
        ("ch", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5),
                               min_df=3, sublinear_tf=True)),
        ("lex", LexiconFeatures()),
    ])
    return Pipeline([("feats", features),
                     ("clf", LogisticRegression(max_iter=3000, class_weight="balanced",
                                                random_state=42))])

### Train/Test Split

In [4]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
grid = GridSearchCV(build_pipeline(), {"clf__C": [0.5, 1.0, 3.0, 10.0]},
                    cv=StratifiedKFold(4, shuffle=True, random_state=42),
                    scoring="f1_macro", n_jobs=1)          # ← was -1
grid.fit(X_tr, y_tr)
print("best C:", grid.best_params_["clf__C"], " inner CV macro-F1: %.3f" % grid.best_score_)

best C: 0.5  inner CV macro-F1: 0.364


### Vectorize with TF-IDF

In [5]:
classes = grid.best_estimator_.named_steps["clf"].classes_
proba_te = grid.best_estimator_.predict_proba(X_te)
ev_te = (proba_te * classes.astype(float)).sum(axis=1)

print("===== HELD-OUT: star classification =====")
print(classification_report(y_te, grid.best_estimator_.predict(X_te), zero_division=0))
print("Rating MAE: %.3f | range %.2f-%.2f" % (mean_absolute_error(y_te, ev_te), ev_te.min(), ev_te.max()))

sent_true = [rating_to_sentiment(v) for v in y_te]
sent_pred = [probs_to_outputs(p, classes)[1] for p in proba_te]
print("===== HELD-OUT: sentiment =====")
print(classification_report(sent_true, sent_pred, zero_division=0))

===== HELD-OUT: star classification =====
              precision    recall  f1-score   support

           1       0.47      0.58      0.52        12
           2       0.36      0.57      0.44         7
           3       0.20      0.18      0.19        11
           4       0.36      0.31      0.33        29
           5       0.68      0.66      0.67        59

    accuracy                           0.52       118
   macro avg       0.41      0.46      0.43       118
weighted avg       0.52      0.52      0.51       118

Rating MAE: 0.792 | range 1.59-4.64
===== HELD-OUT: sentiment =====
              precision    recall  f1-score   support

    negative       0.59      0.84      0.70        19
     neutral       0.00      0.00      0.00        11
    positive       0.93      0.94      0.94        88

    accuracy                           0.84       118
   macro avg       0.51      0.60      0.54       118
weighted avg       0.79      0.84      0.81       118



### Train the Logistic Regression classifier

In [6]:
cv = StratifiedKFold(5, shuffle=True, random_state=42)
cv_proba = cross_val_predict(build_pipeline().set_params(clf__C=grid.best_params_["clf__C"]),
                             X, y, cv=cv, method="predict_proba", n_jobs=1)   # ← was -1
cv_classes = np.unique(y)
cv_ev = (cv_proba * cv_classes.astype(float)).sum(axis=1)
cv_sent = [probs_to_outputs(p, cv_classes)[1] for p in cv_proba]
cv_true = [rating_to_sentiment(v) for v in y]
print("5-FOLD CV  sentiment macro-F1: %.3f | acc: %.3f | rating MAE: %.3f"
      % (f1_score(cv_true, cv_sent, average="macro"),
         accuracy_score(cv_true, cv_sent), mean_absolute_error(y, cv_ev)))

5-FOLD CV  sentiment macro-F1: 0.601 | acc: 0.847 | rating MAE: 0.759


### Cross-validate

In [7]:
final = build_pipeline().set_params(clf__C=grid.best_params_["clf__C"])
final.fit(X, y)                                   # refit on 100% before saving
os.makedirs(MODELS_DIR, exist_ok=True)
joblib.dump({"pipeline": final, "classes": final.named_steps["clf"].classes_,
             "best_C": grid.best_params_["clf__C"]},
            os.path.join(MODELS_DIR, "review_model.pkl"))
print("saved models/review_model.pkl")

# NEGATION SANITY CHECK — these pairs MUST differ now
for a, b in [("the food was tasty", "the food was not tasty"),
             ("the ambience was good", "the ambience was not good")]:
    sa = probs_to_outputs(final.predict_proba([clean_review_text(a)])[0], final.named_steps["clf"].classes_)
    sb = probs_to_outputs(final.predict_proba([clean_review_text(b)])[0], final.named_steps["clf"].classes_)
    print(f"{a:30s} -> {sa[1]:8s} {sa[0]:.2f}")
    print(f"{b:30s} -> {sb[1]:8s} {sb[0]:.2f}")
    print("DIFFERENT ✔" if sa[1] != sb[1] or abs(sa[0]-sb[0]) > 0.3 else "** STILL BROKEN **")

saved models/review_model.pkl
the food was tasty             -> positive 3.96
the food was not tasty         -> negative 2.02
DIFFERENT ✔
the ambience was good          -> positive 4.07
the ambience was not good      -> negative 2.08
DIFFERENT ✔


### Inspect which words drive each prediction

### Save the model + vectorizer